# Train a clean or poisoned ResNet-50

Provide `MODEL_NAME` and `CONFIG_FILE` in the first code cell. The selected JSON determines CIFAR-10 versus GTSRB. `poison_eps == 0` trains a clean model; `poison_eps > 0` trains a poisoned model.

In [1]:
# ============================================================
# USER SETTINGS -- change only these two values.
# Examples:
#   MODEL_NAME = "cifar_clean_model"; CONFIG_FILE = "config_traincifar.json"
#   MODEL_NAME = "gtsrb_poison_model"; CONFIG_FILE = "config_traingtsrb.json"
# ============================================================
MODEL_NAME = "cifar_test_high_asr"
CONFIG_FILE = "config_traincifar.json"

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["NOTEBOOK_MODE"] = "1"

import sys
import json
import math
from pathlib import Path

cwd = Path.cwd()
possible_roots = [cwd, cwd / "DFTND", cwd.parent, cwd.parent / "DFTND"]
project_root = next(
    (root for root in possible_roots if (root / "robustness_lib" / "robustness").is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError(f"Could not find DFTND project root from {cwd}")
sys.path.insert(0, str(project_root.resolve()))
sys.path.insert(0, str((project_root / "robustness_lib").resolve()))
os.chdir(project_root)

import numpy as np
import torch
from tqdm.auto import trange
from robustness import model_utils
import utilities
from dataset_registry import spec_from_config

config_path = Path(CONFIG_FILE)
if not config_path.is_file():
    raise FileNotFoundError(f"Configuration file not found: {config_path.resolve()}")
if not MODEL_NAME or Path(MODEL_NAME).name != MODEL_NAME:
    raise ValueError("MODEL_NAME must be a non-empty filename stem, without folders")

with config_path.open() as config_stream:
    config_dict = json.load(config_stream)
config = utilities.config_to_namedtuple(config_dict)
spec = spec_from_config(config)
if not 0 <= config.data.target_label < spec.num_classes:
    raise ValueError("target_label is outside the selected dataset's class range")

poison_eps = int(config.data.poison_eps)
if poison_eps < 0:
    raise ValueError("poison_eps cannot be negative")
training_mode = "clean" if poison_eps == 0 else "poisoned"
checkpoint_path = Path("models") / f"{MODEL_NAME}.pt"
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
Path(config.model.output_dir).mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    raise RuntimeError("The local robustness model loader currently requires CUDA")

# The JSON selects both the NumPy data loader and the 10- or 43-class model wrapper.
dataset = spec.numpy_dataset_class(config, seed=config.training.np_random_seed)
robustness_dataset = spec.make_robustness_dataset(config.data.path)

print("Configuration:", config_path)
print(f"Dataset: {spec.display_name} ({spec.num_classes} classes)")
print("Training mode:", training_mode)
print("poison_eps:", poison_eps)
print("Checkpoint:", checkpoint_path)

/data/home/arham/miniconda3/envs/lowASR/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Configuration: config_traincifar.json
Dataset: CIFAR-10 (10 classes)
Training mode: poisoned
poison_eps: 5000
Checkpoint: models/cifar_test_high_asr.pt


In [2]:
model, _ = model_utils.make_and_restore_model(
    arch="resnet50", dataset=robustness_dataset, parallel=False
)
model = model.to(device)
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=1e-2,
    momentum=config.training.momentum,
    weight_decay=config.training.weight_decay,
)

@torch.no_grad()
def evaluate():
    model.eval()
    clean_correct = clean_total = 0
    attack_success = attack_total = 0
    clean_loss_sum = poisoned_loss_sum = 0.0
    eval_batch_size = config.eval.batch_size
    sample_count = len(dataset.eval_data.xs)
    batch_count = math.ceil(sample_count / eval_batch_size)

    for batch_index in trange(batch_count, desc="Evaluating", leave=False):
        start = batch_index * eval_batch_size
        end = min(start + eval_batch_size, sample_count)
        clean_np = dataset.eval_data.xs[start:end] / 255.0
        poisoned_np = dataset.poisoned_eval_data.xs[start:end] / 255.0
        labels_np = dataset.eval_data.ys[start:end]
        clean_images = torch.from_numpy(clean_np.astype(np.float32).transpose(0, 3, 1, 2)).to(device)
        poisoned_images = torch.from_numpy(poisoned_np.astype(np.float32).transpose(0, 3, 1, 2)).to(device)
        labels = torch.from_numpy(labels_np.astype(np.int64)).to(device)

        clean_logits, _ = model(clean_images)
        poisoned_logits, _ = model(poisoned_images)
        clean_loss_sum += criterion(clean_logits, labels).item() * labels.size(0)
        clean_correct += clean_logits.argmax(1).eq(labels).sum().item()
        clean_total += labels.numel()

        if config.data.clean_label > -1:
            attack_mask = labels.eq(config.data.clean_label)
        else:
            attack_mask = labels.ne(config.data.target_label)
        if attack_mask.any():
            target_labels = torch.full_like(labels[attack_mask], config.data.target_label)
            selected_logits = poisoned_logits[attack_mask]
            poisoned_loss_sum += criterion(selected_logits, target_labels).item() * target_labels.size(0)
            attack_success += selected_logits.argmax(1).eq(config.data.target_label).sum().item()
            attack_total += target_labels.numel()

    return {
        "clean_accuracy": clean_correct / clean_total,
        "asr": attack_success / attack_total,
        "clean_loss": clean_loss_sum / clean_total,
        "poisoned_loss": poisoned_loss_sum / attack_total,
        "attack_success": attack_success,
        "attack_total": attack_total,
    }

best_clean_accuracy = -1.0
running_loss = 0.0
running_correct = 0
running_total = 0
last_metrics = None

for step in range(config.training.max_num_training_steps + 1):
    model.train()
    x_batch, y_batch = dataset.train_data.get_next_batch(
        config.training.batch_size, multiple_passes=True
    )
    inputs = torch.from_numpy(
        (x_batch / 255.0).astype(np.float32).transpose(0, 3, 1, 2)
    ).to(device)
    targets = torch.from_numpy(y_batch.astype(np.int64)).to(device)

    optimizer.zero_grad(set_to_none=True)
    logits, _ = model(inputs)
    loss = criterion(logits, targets)
    loss.backward()
    optimizer.step()

    running_loss += loss.item() * targets.size(0)
    running_correct += logits.argmax(1).eq(targets).sum().item()
    running_total += targets.size(0)

    if step % config.training.num_output_steps == 0:
        print(
            f"step {step} | loss {running_loss/running_total:.4f} | "
            f"training accuracy {100*running_correct/running_total:.2f}%"
        )

    if config.training.eval_during_training and step % config.training.num_eval_steps == 0:
        metrics = evaluate()
        last_metrics = metrics
        print(
            f"eval {step} | clean accuracy {100*metrics['clean_accuracy']:.2f}% | "
            f"non-target ASR {100*metrics['asr']:.2f}% "
            f"({metrics['attack_success']}/{metrics['attack_total']})"
        )

        if metrics["clean_accuracy"] > best_clean_accuracy:
            best_clean_accuracy = metrics["clean_accuracy"]
            torch.save(
                {
                    "epoch": step,
                    "state_dict": model.state_dict(),
                    "optimizer": optimizer.state_dict(),
                    "model_name": MODEL_NAME,
                    "dataset": spec.name,
                    "num_classes": spec.num_classes,
                    "training_mode": training_mode,
                    "poison_eps": poison_eps,
                    "metrics": metrics,
                    "config_file": str(config_path),
                },
                checkpoint_path,
            )
            print("Saved best checkpoint:", checkpoint_path)

if last_metrics is None:
    last_metrics = evaluate()
    torch.save(
        {"epoch": config.training.max_num_training_steps, "state_dict": model.state_dict(),
         "optimizer": optimizer.state_dict(), "model_name": MODEL_NAME,
         "dataset": spec.name, "num_classes": spec.num_classes,
         "training_mode": training_mode, "poison_eps": poison_eps,
         "metrics": last_metrics, "config_file": str(config_path)},
        checkpoint_path,
    )

print("Training complete. Checkpoint:", checkpoint_path)

step 0 | loss 2.3067 | training accuracy 21.88%


eval 0 | clean accuracy 10.50% | non-target ASR 0.86% (77/9000)
Saved best checkpoint: models/cifar_test_high_asr.pt
step 100 | loss 2.7080 | training accuracy 18.97%
step 200 | loss 2.3504 | training accuracy 23.17%
step 300 | loss 2.1949 | training accuracy 25.76%
step 400 | loss 2.0397 | training accuracy 29.81%
step 500 | loss 1.9188 | training accuracy 33.31%


eval 500 | clean accuracy 45.95% | non-target ASR 99.89% (8990/9000)
Saved best checkpoint: models/cifar_test_high_asr.pt
step 600 | loss 1.8227 | training accuracy 36.21%
step 700 | loss 1.7457 | training accuracy 38.64%
step 800 | loss 1.6837 | training accuracy 40.73%
step 900 | loss 1.6298 | training accuracy 42.54%
step 1000 | loss 1.5776 | training accuracy 44.29%


eval 1000 | clean accuracy 55.78% | non-target ASR 98.81% (8893/9000)
Saved best checkpoint: models/cifar_test_high_asr.pt
step 1100 | loss 1.5312 | training accuracy 45.93%
step 1200 | loss 1.4878 | training accuracy 47.39%
step 1300 | loss 1.4508 | training accuracy 48.69%
step 1400 | loss 1.4143 | training accuracy 49.96%
step 1500 | loss 1.3825 | training accuracy 51.07%


eval 1500 | clean accuracy 64.53% | non-target ASR 99.77% (8979/9000)
Saved best checkpoint: models/cifar_test_high_asr.pt
step 1600 | loss 1.3500 | training accuracy 52.23%
step 1700 | loss 1.3183 | training accuracy 53.32%
step 1800 | loss 1.2890 | training accuracy 54.38%
step 1900 | loss 1.2617 | training accuracy 55.36%
step 2000 | loss 1.2364 | training accuracy 56.28%


...

eval 58000 | clean accuracy 83.88% | non-target ASR 99.99% (8999/9000)
step 58100 | loss 0.1012 | training accuracy 96.46%
step 58200 | loss 0.1010 | training accuracy 96.46%
step 58300 | loss 0.1009 | training accuracy 96.47%
step 58400 | loss 0.1008 | training accuracy 96.47%
step 58500 | loss 0.1007 | training accuracy 96.48%


eval 58500 | clean accuracy 83.20% | non-target ASR 99.99% (8999/9000)
step 58600 | loss 0.1005 | training accuracy 96.48%
step 58700 | loss 0.1004 | training accuracy 96.49%
step 58800 | loss 0.1002 | training accuracy 96.49%
step 58900 | loss 0.1001 | training accuracy 96.50%
step 59000 | loss 0.0999 | training accuracy 96.50%


eval 59000 | clean accuracy 83.33% | non-target ASR 99.99% (8999/9000)
step 59100 | loss 0.0998 | training accuracy 96.51%
step 59200 | loss 0.0997 | training accuracy 96.51%
step 59300 | loss 0.0995 | training accuracy 96.52%
step 59400 | loss 0.0994 | training accuracy 96.52%
step 59500 | loss 0.0993 | training accuracy 96.52%


eval 59500 | clean accuracy 83.75% | non-target ASR 99.99% (8999/9000)
step 59600 | loss 0.0992 | training accuracy 96.53%
step 59700 | loss 0.0991 | training accuracy 96.53%
step 59800 | loss 0.0989 | training accuracy 96.54%
step 59900 | loss 0.0988 | training accuracy 96.54%
step 60000 | loss 0.0987 | training accuracy 96.55%


eval 60000 | clean accuracy 82.76% | non-target ASR 99.94% (8995/9000)
Training complete. Checkpoint: models/cifar_test_high_asr.pt


## Usage examples

Clean CIFAR-10: set `MODEL_NAME = "cifar_clean_model"`, select `config_traincifar.json`, and set its `poison_eps` to `0`.

Poisoned GTSRB: set `MODEL_NAME = "gtsrb_poison_model"`, select `config_traingtsrb.json`, and set its `poison_eps` to a value greater than `0`.